In [1]:
import geemap
import ee
import os
import numpy as np

In [2]:
ee.Authenticate()
ee.Initialize()

In [3]:
base = os.path.join(os.getcwd(),'..')
outPath = os.path.join(base,'data','gpp')
os.makedirs(outPath, exist_ok = True)

In [4]:
shapeFile = os.path.join(os.getcwd(),'..','shape','States_shapefile.shp')
shape = geemap.shp_to_ee(shapeFile)

In [5]:
def stats_download(image,statsFile,scale,shape):
    if not os.path.exists(statsFile):
        geemap.zonal_stats(image, shape, statsFile, stats_type='mean', scale=scale)

In [11]:
def get_gpp(y,crop,shape,m):
    y = int(y)
    m = int(m)
    gppCol = ee.ImageCollection('projects/pml_evapotranspiration/PML/OUTPUT/PML_V22a').filterBounds(shape).filter(ee.Filter.calendarRange(y, y, 'year')).filter(ee.Filter.calendarRange(m, m, 'month'))
    gpp_masked = gppCol.select('GPP').sum().updateMask(crop).multiply(0.01)
    return gpp_masked

In [12]:
def get_area(y,outPath,shape,name):
    start_date = str(y)+'-01-01'
    end_date = str(y)+'-12-31'
    crop_imageColl = ee.ImageCollection('USDA/NASS/CDL').filter(ee.Filter.date(start_date,end_date)).filterBounds(shape)
    crop_image = crop_imageColl.select('cropland').first()
    crop = crop_image.eq(1)
    for m in np.arange(1,13):
        loc_c = os.path.join(outPath, str(y)+'_'+str(m)+'_'+name+'.csv')
        gpp = get_gpp(y,crop,shape,m)
        stats_download(gpp,loc_c,30,shape)
    

In [13]:
for d in range(2010,2016):
    get_area(d,outPath,shape,'Corn')
    

Computing statistics ...
